In [9]:
class Position:
    def __init__(self):
        # Simple representation: pieces are just values at squares
        # squares 0-63 (like a chess board)
        # positive = white pieces, negative = black pieces
        self.board = [0] * 64
        self.white_to_move = True
        
        # Add some pieces for testing
        # White: queen(9) on square 10, knight(3) on square 20
        # Black: queen(9) on square 54, bishop(3) on square 44
        self.board[10] = 9   # White queen
        self.board[20] = 3   # White knight
        self.board[54] = -9  # Black queen
        self.board[44] = -3  # Black bishop
    
    def evaluate(self):
        """Sum of material: positive = white ahead"""
        material_sum = sum(self.board)
        return material_sum if self.white_to_move else -material_sum
    
    def get_forcing_moves(self):
        """Generate only captures (forcing moves)"""
        moves = []
        
        # Determine whose pieces to move
        moving_side = 1 if self.white_to_move else -1
        
        # For each piece of the moving side
        for from_sq in range(64):
            piece = self.board[from_sq]
            
            # Is this our piece?
            if piece == 0 or piece * moving_side < 0:  # Not our piece
                continue
            
            # Try to capture every other piece
            for to_sq in range(64):
                target = self.board[to_sq]
                
                # Can only capture opponent pieces
                if target == 0 or target * moving_side > 0:  # Empty or our piece
                    continue
                
                # This is a valid capture
                moves.append({
                    'from': from_sq,
                    'to': to_sq,
                    'piece': piece,
                    'captured': target
                })
        
        return moves
    
    def make_move(self, move):
        """Apply a move to the board"""
        from_sq = move['from']
        to_sq = move['to']
        
        # Move the piece
        self.board[to_sq] = self.board[from_sq]
        self.board[from_sq] = 0
        
        # Switch turns
        self.white_to_move = not self.white_to_move
    
    def undo_move(self, move):
        """Undo a move"""
        from_sq = move['from']
        to_sq = move['to']
        
        # Restore the piece
        self.board[from_sq] = move['piece']
        
        # Restore the captured piece
        self.board[to_sq] = move['captured']
        
        # Switch turns back
        self.white_to_move = not self.white_to_move
    
    def display(self):
        """Show the board state"""
        print("\nBoard state:")
        for i in range(64):
            if self.board[i] != 0:
                piece_char = "KQRBNP"[abs(self.board[i]) - 1] if abs(self.board[i]) <= 6 else str(abs(self.board[i]))
                side = "W" if self.board[i] > 0 else "B"
                print(f"  Sq {i:2d}: {side}{piece_char} (value: {self.board[i]:2d})")
        
        print(f"  Eval: {self.evaluate():3d}")
        print(f"  To move: {'White' if self.white_to_move else 'Black'}")

In [16]:
def quiescent(position, alpha, beta, is_maximizing, depth=0, max_depth=10):
    current_eval = position.evaluate()
    forcing_moves = position.get_forcing_moves()
    if depth >= max_depth or not forcing_moves:
        return current_eval
        
    if is_maximizing:
        if current_eval >= beta:
            return beta
        alpha = max(alpha, current_eval)
    else:
        if current_eval <= alpha:
            return alpha
        beta = min(beta, current_eval)

    forcing_moves.sort(key=lambda m: 100 * abs(m['captured']) - abs(m['piece']), reverse=True)
    best_eval = current_eval

    for move in forcing_moves:
        position.make_move(move)
        eval_score = quiescent(position, alpha, beta, not is_maximizing, depth + 1, max_depth)
        position.undo_move(move)
        if is_maximizing:
            best_eval = max(best_eval, eval_score)
            alpha = max(alpha, best_eval)
            if alpha >= beta:
                break;
        else:
            best_eval = min(best_eval, eval_score)
            beta = min(beta, best_eval)
            if beta <= alpha:
                break
    return best_eval

In [17]:
pos = Position()
print("Initial position:")
pos.display()

result = quiescent(pos, alpha=-1000, beta=1000, is_maximizing=True)
print(f"\nQuiescent evaluation: {result}")
print("\nFinal board state:")
pos.display()

Initial position:

Board state:
  Sq 10: W9 (value:  9)
  Sq 20: WR (value:  3)
  Sq 44: BR (value: -3)
  Sq 54: B9 (value: -9)
  Eval:   0
  To move: White

Quiescent evaluation: 0

Final board state:

Board state:
  Sq 10: W9 (value:  9)
  Sq 20: WR (value:  3)
  Sq 44: BR (value: -3)
  Sq 54: B9 (value: -9)
  Eval:   0
  To move: White
